<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">Comprehensive Tool Calling Training Guide</h1>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">This notebook provides a comprehensive guide to tool calling training using LLaMA-Factory, covering:</p>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Tool Calling Fundamentals</strong>: Understanding function calling and tool integration</li>
<li style="margin:6px 0;"><strong>Dataset Preparation</strong>: Tool calling data formatting and examples</li>
<li style="margin:6px 0;"><strong>Training Configurations</strong>: LoRA, QLoRA, and full fine-tuning for tool calling</li>
<li style="margin:6px 0;"><strong>Model Training</strong>: Training models to use tools effectively</li>
<li style="margin:6px 0;"><strong>Evaluation</strong>: Tool calling accuracy and performance assessment</li>
<li style="margin:6px 0;"><strong>Advanced Techniques</strong>: Multi-tool scenarios and complex tool chains</li>
<li style="margin:6px 0;"><strong>Best Practices</strong>: Optimization and deployment strategies</li>
</ol>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Table of Contents</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#setup-and-installation">Setup and Installation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#tool-calling-fundamentals">Tool Calling Fundamentals</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#dataset-preparation">Dataset Preparation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#training-configurations">Training Configurations</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#model-training">Model Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#evaluation-and-testing">Evaluation and Testing</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#advanced-techniques">Advanced Techniques</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#best-practices">Best Practices</a></li>
</ul>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Setup and Installation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">First, let's install the required dependencies and set up the environment for tool calling.</p>
</div>


In [ ]:
# Install dependencies for tool calling
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers[torch] datasets jsonlines openai
%pip install llamafactory peft accelerate

# Import required libraries
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from llamafactory import ChatModel
import json
import os
import yaml
from typing import List, Dict, Any

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Tool Calling Fundamentals</h2>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">What is Tool Calling?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Tool calling (function calling) is the ability of language models to:</p>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Understand</strong> when and how to use external tools</li>
<li style="margin:6px 0;"><strong>Generate</strong> properly formatted tool calls with correct parameters</li>
<li style="margin:6px 0;"><strong>Interpret</strong> tool responses and continue conversations</li>
<li style="margin:6px 0;"><strong>Chain</strong> multiple tools together for complex tasks</li>
</ol>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Tool Definition Format</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Tools are defined using JSON Schema with the following structure:</p>
<pre style="background:#1b2330;color:#e6edf3;padding:14px 16px;border-radius:14px;border:1px solid #334155;overflow-x:auto;line-height:1.7;margin:14px 0;"><code class="language-json" style="background:transparent;color:#e6edf3;border:0;padding:0;">{
  &quot;name&quot;: &quot;tool_name&quot;,
  &quot;description&quot;: &quot;What this tool does&quot;,
  &quot;parameters&quot;: {
    &quot;type&quot;: &quot;object&quot;,
    &quot;properties&quot;: {
      &quot;param1&quot;: {&quot;type&quot;: &quot;string&quot;, &quot;description&quot;: &quot;Parameter description&quot;},
      &quot;param2&quot;: {&quot;type&quot;: &quot;number&quot;, &quot;description&quot;: &quot;Another parameter&quot;}
    },
    &quot;required&quot;: [&quot;param1&quot;]
  }
}
</code></pre>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Dataset Preparation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Let's prepare datasets for tool calling training with various types of tools.</p>
</div>


In [ ]:
# 1. Weather Information Tool Dataset
weather_tool_data = [
    {
        "conversations": [
            {"from": "human", "value": "What's the weather like in New York today?"},
            {"from": "gpt", "value": "First, the user is asking about the weather in New York. I need to use the get_weather tool to fetch current weather information."}
        ],
        "tools": [
            {
                "name": "get_weather",
                "description": "Get current weather information for a city",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name"}
                    },
                    "required": ["city"]
                }
            }
        ]
    },
    {
        "conversations": [
            {"from": "human", "value": "Is it going to rain in London tomorrow?"},
            {"from": "gpt", "value": "To answer about rain in London tomorrow, I should use the get_weather_forecast tool."}
        ],
        "tools": [
            {
                "name": "get_weather_forecast",
                "description": "Get weather forecast for a city",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name"},
                        "days": {"type": "integer", "description": "Number of days ahead"}
                    },
                    "required": ["city"]
                }
            }
        ]
    }
]

# Save weather tool dataset
with open('data/tool_calling_weather_demo.json', 'w') as f:
    json.dump(weather_tool_data, f, indent=2)


In [ ]:
# 2. Calculator and Math Tools Dataset
math_tool_data = [
    {
        "conversations": [
            {"from": "human", "value": "Calculate 15 factorial (15!)."},
            {"from": "gpt", "value": "The user wants to calculate 15 factorial. I should use the factorial calculator tool."}
        ],
        "tools": [
            {
                "name": "calculate_factorial",
                "description": "Calculate the factorial of a number",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "number": {"type": "integer", "description": "Number to calculate factorial for"}
                    },
                    "required": ["number"]
                }
            }
        ]
    },
    {
        "conversations": [
            {"from": "human", "value": "What's the square root of 144?"},
            {"from": "gpt", "value": "For square root calculation, I'll use the math_square_root tool."}
        ],
        "tools": [
            {
                "name": "math_square_root",
                "description": "Calculate the square root of a number",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "number": {"type": "number", "description": "Number to calculate square root for"}
                    },
                    "required": ["number"]
                }
            }
        ]
    }
]

# Save math tool dataset
with open('data/tool_calling_math_demo.json', 'w') as f:
    json.dump(math_tool_data, f, indent=2)


In [ ]:
# 3. Multi-Tool Dataset (Complex Scenarios)
multi_tool_data = [
    {
        "conversations": [
            {"from": "human", "value": "I need to calculate the compound interest on $1000 invested for 5 years at 5% annual interest. Can you help?"},
            {"from": "gpt", "value": "This requires compound interest calculation. I'll use the financial calculator tool."}
        ],
        "tools": [
            {
                "name": "calculate_compound_interest",
                "description": "Calculate compound interest",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "principal": {"type": "number", "description": "Initial amount"},
                        "rate": {"type": "number", "description": "Annual interest rate (decimal)"},
                        "time": {"type": "number", "description": "Time in years"},
                        "compounds": {"type": "integer", "description": "Compounding frequency per year"}
                    },
                    "required": ["principal", "rate", "time"]
                }
            }
        ]
    },
    {
        "conversations": [
            {"from": "human", "value": "Search for recent news about artificial intelligence developments."},
            {"from": "gpt", "value": "The user wants recent AI news. I should use the web search tool."}
        ],
        "tools": [
            {
                "name": "web_search",
                "description": "Search the web for information",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search query"},
                        "max_results": {"type": "integer", "description": "Maximum number of results"}
                    },
                    "required": ["query"]
                }
            }
        ]
    }
]

# Save multi-tool dataset
with open('data/tool_calling_multi_demo.json', 'w') as f:
    json.dump(multi_tool_data, f, indent=2)


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Training Configurations</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Let's create training configurations for tool calling scenarios.</p>
</div>


In [ ]:
# 1. LoRA Tool Calling Configuration
lora_tool_config = {
    "model_name_or_path": "meta-llama/Meta-Llama-3-8B-Instruct",
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_rank": 16,
    "lora_alpha": 64,
    "lora_dropout": 0.1,
    "lora_target": "all",
    "dataset": "tool_calling_weather_demo",
    "template": "llama3",
    "cutoff_len": 2048,
    "max_samples": 1000,
    "output_dir": "saves/llama3-8b/lora/tool_calling",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1.0e-4,
    "num_train_epochs": 3.0,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "bf16": True,
    "logging_steps": 10,
    "save_steps": 500,
    "plot_loss": True,
    "overwrite_output_dir": True
}

with open('examples/train_lora/llama3_lora_tool_calling_demo.yaml', 'w') as f:
    yaml.dump(lora_tool_config, f, default_flow_style=False)
